In [ ]:
import sys; sys.path.append("../../"); sys.path.append("../../../.."); sys.path.append("../../../gmsh/"); sys.path.append("../../experiments/"); sys.path.append("../../../")
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

In [ ]:

amplitudes = np.linspace(0, 0.4, 9)[:7]
# amplitudes = [amplitudes[0]]
radius = np.linspace(0.5, 2.4, 20)[:-1]
angles = np.linspace(0, 90, 31)

In [ ]:
amplitudes, radius, angles

In [ ]:
variable = '0.30_2.10_21.00'

In [ ]:
m = MeshFEM.mesh.Mesh('../../experiments/output/mirror_cosine_dash/2023_05_19_10_43/mesh_mirror_cosine_dash_{}.obj'.format(variable))
vx = m.vertices()
new_vx = np.zeros((vx.shape[0], 3))
new_vx[:, :2] = vx
m = MeshFEM.Mesh(new_vx, m.elements())
fusedVtx = np.load('../../experiments/output/mirror_cosine_dash/2023_05_19_10_43/fusedVtx_mirror_cosine_dash_{}.npy'.format(variable))

In [ ]:
visualization.plot_2d_mesh(m, pointList = fusedVtx, width = 5, height = 5)

In [ ]:
benchmark.reset()
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)
benchmark.report()

In [ ]:

az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, True, False)

dof = np.load("../../experiments/output/mirror_cosine_dash/2023_05_19_10_43/high_pressure_dofs_mirror_cosine_dash_{}.npy".format(variable))
az_ipu.setVars(dof)

In [ ]:
viewer = TriMeshViewer(az_ipu, width=768, height=640)
viewer.showWireframe(False)
viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
name = "mirror_cosine_dash"
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
result_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  

In [ ]:
render = viewer.offscreenRenderer(1000, 1000)
render.render()
render.save("{}/{}_render_{}_{}.png".format(result_folder, '', name, variable))

In [ ]:
import importlib, periodic_simulation_setup

In [ ]:
importlib.reload(periodic_simulation_setup)

In [ ]:
points = periodic_simulation_setup.visualize_average_deformation_gradient(az_ipu.ipu, 100, plot_max_r=1, plot_min_r=0, show_figure=True, filename = "{}/average_deformation_gradient_{}_{}.svg".format(result_folder, name, variable))

In [ ]:
allowBending = False

In [ ]:
if not allowBending:
    fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6
opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
# cr = az_optimizer.optimize()

In [ ]:
stiffness_shift = 1e-15
success = False
for i in range(15):
    try:
        stiffness_values, sampled_alphas, stiffness_coefficient = periodic_simulation_setup.visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = stiffness_shift, fixedVars = [], filename = "{}/stiffness_{}_{}.svg".format(result_folder, name, variable), show_figure=True, plot_min_r=0, generate_images = True)
        np.save("{}/stiffness_values_{}_{}.npy".format(result_folder, name, variable), stiffness_values)
        np.save("{}/sampled_alphas_{}_{}.npy".format(result_folder, name, variable), sampled_alphas)
        np.save("{}/stiffness_coefficient_{}_{}.npy".format(result_folder, name, variable), stiffness_coefficient)
        print("Solved using stiffness shift: ", stiffness_shift)
        success = True
        break
    except:
        print("failed to compute stiffness with shift ", stiffness_shift)
        stiffness_shift *= 10
if (not success):
    print("Failed to solve stiffness!")


In [ ]:
low_pressure_tag = "low_pressure"

In [ ]:
np.save("{}/scale_factors_{}_{}.npy".format(result_folder, name, variable), get_deformation_scale_factors(az_ipu.ipu))
np.save("{}/kappa_{}_{}.npy".format(result_folder, name, variable), az_ipu.getVars()[-2])

np.save("{}/average_deformation_gradient_matrix_{}_{}.npy".format(result_folder, name, variable), get_deformation_matrix(az_ipu.ipu))

np.save("{}/{}_strain_values_{}_{}.npy".format(result_folder, low_pressure_tag, name, variable), utils.getStrains(az_ipu.ipu.sheet)[:, 0])

np.save("{}/{}_dofs_{}_{}.npy".format(result_folder, low_pressure_tag, name, variable), az_ipu.getVars())

In [ ]:
viewer.saveObj("{}/mesh_{}_{}.obj".format(result_folder, name, variable))

In [ ]:
viewer.saveColorizedObj("{}/color_mesh_{}_{}.obj".format(result_folder, name, variable))